# COVID-19 Risk Prediction & Pandemic Intelligence Platform

This notebook demonstrates the end-to-end data science workflow for the COVID-19 healthcare analytics platform. It includes:
1. **Data Cleaning**: Handling duplicates, stripping whitespaces, and imputing missing values.
2. **Exploratory Data Analysis (EDA)**: Visualizing demographic distributions, clinical metrics, symptoms, and correlations.
3. **Feature Engineering**: Formatting features, combining clinical cough indicators, and encoding binary inputs.
4. **Model Training & Comparison**: Splitting the dataset, scaling features, and fitting Logistic Regression, Random Forest, and Gradient Boosting models.
5. **Model Evaluation**: Comparing metric performance (F1, Accuracy, Precision, Recall, ROC-AUC) to select the best model.
6. **Inference Pipeline**: Saving artifacts for the Streamlit web application and testing inference on mock data.
7. **Power BI Preparation**: Enriching the clean data with geospatial, seasonal, and health outcome markers.

## 1. Import Libraries

We load standard libraries for data handling, plotting, and machine learning.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import joblib

# Set seaborn style for rich visual charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries imported successfully!")

## 2. Load Dataset

We read the raw dataset from `data/raw_covid_data.csv` and check the initial dimension.

In [ ]:
data_path = "../data/raw_covid_data.csv"
if not os.path.exists(data_path):
    # Fallback to current folder if notebook is run in a different working directory context
    data_path = "data/raw_covid_data.csv"

df = pd.read_csv(data_path)
print(f"Dataset successfully loaded! Dimensions: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

## 3. Dataset Overview

We inspect data types, missing values, duplicates, and general summary statistics.

In [ ]:
print("--- Data Types and Non-Null Counts ---")
df.info()

print("\n--- Null Value Counts ---")
print(df.isnull().sum())

print(f"\nDuplicate rows found: {df.duplicated().sum()}")

print("\n--- Summary Statistics for Numeric Features ---")
df.describe().T

## 4. Data Cleaning

We clean column names by stripping trailing whitespaces, remove duplicate rows, and impute missing values (mean for numericals, mode for categoricals).

In [ ]:
# Clean column names
df.columns = df.columns.str.strip()

# Handle duplicates
dup_count = df.duplicated().sum()
if dup_count > 0:
    df = df.drop_duplicates()
    print(f"Dropped {dup_count} duplicate rows.")

# Impute numeric columns with mean
num_cols = ['Age', 'Temperature_F', 'Oxygen_Level']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        mean_val = df[col].mean()
        df[col] = df[col].fillna(mean_val)

# Impute categorical columns with mode
cat_cols = [col for col in df.columns if col not in num_cols + ['Patient_ID', 'Patient_Name']]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.capitalize()
    mode_val = df[col].mode()[0] if not df[col].empty else 'No'
    df[col] = df[col].replace({'Nan': mode_val, 'None': mode_val, '': mode_val})
    df[col] = df[col].fillna(mode_val)

# Re-validate dimensions
print(f"Post-cleaning dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns.")

## 5. Exploratory Data Analysis (EDA)

We generate demographic breakdowns, symptom plots, oxygen level impact, and feature correlation heatmaps.

In [ ]:
# 1. Age Distribution
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x='Age', kde=True, color='teal', bins=20)
plt.title('Age Distribution of Patients')
plt.xlabel('Age (Years)')
plt.ylabel('Count')
plt.show()

# 2. Gender-wise Positive Count
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Gender', hue='COVID_Result', palette='viridis')
plt.title('COVID Diagnosis by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.show()

# 3. Oxygen Level distribution vs. COVID diagnosis
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='COVID_Result', y='Oxygen_Level', hue='COVID_Result', palette='coolwarm', legend=False)
plt.title('Oxygen Levels vs COVID Diagnosis')
plt.xlabel('COVID Positive')
plt.ylabel('Oxygen Level (SpO2 %)')
plt.show()

# 4. Temperature distribution vs. COVID diagnosis
plt.figure(figsize=(8, 4))
sns.violinplot(data=df, x='COVID_Result', y='Temperature_F', hue='COVID_Result', palette='Oranges', legend=False)
plt.title('Body Temperature vs COVID Diagnosis')
plt.xlabel('COVID Positive')
plt.ylabel('Temperature (°F)')
plt.show()

## 6. Feature Engineering

We combine the symptoms of `Minor_Cough` and `Dry_Cough` into `Cough`, map columns to binary integers (0/1), and prepare the target label.

In [ ]:
# Map Gender: Male -> 1, Female -> 0
df['Gender_Encoded'] = df['Gender'].map({'Male': 1, 'Female': 0}).fillna(0).astype(int)

# Build Cough column
if 'Minor_Cough' in df.columns and 'Dry_Cough' in df.columns:
    df['Cough'] = np.where((df['Minor_Cough'] == 'Yes') | (df['Dry_Cough'] == 'Yes'), 'Yes', 'No')
else:
    df['Cough'] = df.get('Minor_Cough', df.get('Dry_Cough', 'No'))

# Encode other indicators to binary
binary_fields = ['Cough', 'Breathing_Difficulty', 'Heart_Disease', 'Diabetes']
for field in binary_fields:
    df[field + '_Encoded'] = df[field].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

# Encode Target Column (COVID_Result)
df['Target_Encoded'] = df['COVID_Result'].map({'Yes': 1, 'No': 0}).fillna(0).astype(int)

print("Feature mapping completed! Preprocessed DataFrame columns:")
print(df.columns.tolist())

# 5. Correlation Heatmap
plt.figure(figsize=(10, 8))
corr_cols = ['Age', 'Gender_Encoded', 'Temperature_F', 'Oxygen_Level', 
             'Cough_Encoded', 'Breathing_Difficulty_Encoded', 'Heart_Disease_Encoded', 
             'Diabetes_Encoded', 'Target_Encoded']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix of Modeling Features')
plt.show()

## 7. Model Training

We isolate the input features and target, split them into 80% train and 20% test sets, scale numeric dimensions, and train our models.

In [ ]:
# Input Features & Target
feature_cols = [
    'Age', 'Gender_Encoded', 'Temperature_F', 'Oxygen_Level', 
    'Cough_Encoded', 'Breathing_Difficulty_Encoded', 'Heart_Disease_Encoded', 'Diabetes_Encoded'
]

X = df[feature_cols]
y = df['Target_Encoded']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train set shape: {X_train.shape}, Test set shape: {X_test.shape}")

# Scale Numeric Features
scaler = StandardScaler()
numeric_names = ['Age', 'Temperature_F', 'Oxygen_Level']
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_names] = scaler.fit_transform(X_train[numeric_names])
X_test_scaled[numeric_names] = scaler.transform(X_test[numeric_names])

# Initialize classifiers
classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=8),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, learning_rate=0.1, max_depth=4)
}

print("Classifiers initialized.")

## 8. Model Evaluation

We fit each classifier, predict on the test set, and evaluate metrics to select the best option.

In [ ]:
results = {}
best_f1 = 0
best_name = None
best_model = None

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = [acc, prec, rec, f1, auc]
    
    print(f"=== {name} ===")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  ROC-AUC:   {auc:.4f}")
    print(f"  Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n")
    
    if f1 > best_f1:
        best_f1 = f1
        best_name = name
        best_model = clf

print(f"Best model by F1-Score: {best_name} (F1: {best_f1:.4f})")

# Create comparative metrics table
metrics_df = pd.DataFrame.from_dict(results, orient='index', columns=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'])
metrics_df

## 9. Save Best Model & Preprocessing Pipeline

We save the preprocessor scaler and features layout to `preprocessing_pipeline.pkl` and the classifier model to `covid_risk_model.pkl`.

In [ ]:
# Preprocessor configuration
preprocessing_pipeline = {
    'scaler': scaler,
    'feature_cols': feature_cols,
    'numeric_names': numeric_names,
    'gender_mapping': {'Male': 1, 'Female': 0},
    'binary_mapping': {'Yes': 1, 'No': 0}
}

os.makedirs('../models', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Save preprocessing configurations
try:
    joblib.dump(preprocessing_pipeline, '../models/preprocessing_pipeline.pkl')
    joblib.dump(best_model, '../models/covid_risk_model.pkl')
except FileNotFoundError:
    joblib.dump(preprocessing_pipeline, 'models/preprocessing_pipeline.pkl')
    joblib.dump(best_model, 'models/covid_risk_model.pkl')

print("Model and preprocessing objects successfully serialized to models/ folder.")

## 10. Test Prediction Example

We mock patient data to run an inference walkthrough.

In [ ]:
# Load pipelines
try:
    pipeline_data = joblib.load('../models/preprocessing_pipeline.pkl')
    predictor = joblib.load('../models/covid_risk_model.pkl')
except FileNotFoundError:
    pipeline_data = joblib.load('models/preprocessing_pipeline.pkl')
    predictor = joblib.load('models/covid_risk_model.pkl')

# Mock input data representing a patient
mock_patient = {
    'Age': 58,
    'Gender': 'Male',
    'Temperature_F': 101.8,
    'Oxygen_Level': 89,
    'Cough': 'Yes',
    'Breathing_Difficulty': 'Yes',
    'Heart_Disease': 'No',
    'Diabetes': 'Yes'
}

# Encode the mock inputs to match feature columns
encoded_input = {}
encoded_input['Age'] = mock_patient['Age']
encoded_input['Gender_Encoded'] = pipeline_data['gender_mapping'][mock_patient['Gender']]
encoded_input['Temperature_F'] = mock_patient['Temperature_F']
encoded_input['Oxygen_Level'] = mock_patient['Oxygen_Level']
encoded_input['Cough_Encoded'] = pipeline_data['binary_mapping'][mock_patient['Cough']]
encoded_input['Breathing_Difficulty_Encoded'] = pipeline_data['binary_mapping'][mock_patient['Breathing_Difficulty']]
encoded_input['Heart_Disease_Encoded'] = pipeline_data['binary_mapping'][mock_patient['Heart_Disease']]
encoded_input['Diabetes_Encoded'] = pipeline_data['binary_mapping'][mock_patient['Diabetes']]

# Format as DataFrame
input_df = pd.DataFrame([encoded_input])

# Scale numeric variables
num_features = pipeline_data['numeric_names']
input_df[num_features] = pipeline_data['scaler'].transform(input_df[num_features])

# Run prediction
risk_prob = predictor.predict_proba(input_df[pipeline_data['feature_cols']])[:, 1][0]
risk_pct = round(risk_prob * 100, 1)

# Risk Category allocation
risk_cat = 'Low Risk'
if risk_pct >= 70 or mock_patient['Oxygen_Level'] < 90:
    risk_cat = 'High Risk'
elif risk_pct >= 40:
    risk_cat = 'Medium Risk'

print("--- Prediction Results ---")
print(f"COVID Risk Probability: {risk_pct}%")
print(f"Patient Severity Status: {risk_cat}")

## 11. Export Power BI Dataset

We enrich the dataset for dashboard visualizations by adding geographic coordinates, registration dates, case outcomes, and case tracking metrics.

In [ ]:
pbi_df = df.copy()

# Risk percentage mapping
X_full = pbi_df[feature_cols].copy()
X_full[numeric_names] = scaler.transform(X_full[numeric_names])
pbi_df['COVID_Risk_Percentage'] = (best_model.predict_proba(X_full)[:, 1] * 100).round(1)

# Categorize Risk levels
pbi_df['Risk_Category'] = np.select(
    [
        (pbi_df['COVID_Risk_Percentage'] < 40),
        (pbi_df['COVID_Risk_Percentage'] >= 40) & (pbi_df['COVID_Risk_Percentage'] < 70),
        (pbi_df['COVID_Risk_Percentage'] >= 70)
    ],
    ['Low Risk', 'Medium Risk', 'High Risk'],
    default='Medium Risk'
)

# Apply safety override clinical indicators
pbi_df.loc[(pbi_df['Oxygen_Level'] < 90) | (pbi_df['Temperature_F'] >= 102.0), 'Risk_Category'] = 'High Risk'
pbi_df.loc[
    ((pbi_df['Oxygen_Level'] >= 90) & (pbi_df['Oxygen_Level'] <= 93) | (pbi_df['Temperature_F'] >= 100.5)) & 
    (pbi_df['Risk_Category'] == 'Low Risk'), 'Risk_Category'
] = 'Medium Risk'

# Recovery & Death mapping based on severity
np.random.seed(42)
outcomes = []
for idx, row in pbi_df.iterrows():
    if row['Target_Encoded'] == 1:
        is_critical = (row['Oxygen_Level'] < 88) or (row['Age'] > 75 and (row['Heart_Disease'] == 'Yes' or row['Diabetes'] == 'Yes'))
        if is_critical:
            outcomes.append('Deceased' if np.random.rand() < 0.35 else 'Recovered')
        else:
            outcomes.append('Deceased' if np.random.rand() < 0.005 else 'Recovered')
    else:
        outcomes.append('Not Applicable')

pbi_df['Patient_Outcome'] = outcomes

# State assigning
states = ['California', 'Texas', 'New York', 'Florida', 'Illinois', 'Pennsylvania', 'Ohio', 'Georgia', 'North Carolina', 'Michigan']
pbi_df['State'] = np.random.choice(states, size=len(pbi_df))

# Dates assigning
dates = pd.date_range(start='2026-01-01', end='2026-06-21', freq='D')
pbi_df['Registration_Date'] = np.random.choice(dates, size=len(pbi_df))
pbi_df['Registration_Month'] = pbi_df['Registration_Date'].dt.strftime('%B')
pbi_df['Registration_Year'] = pbi_df['Registration_Date'].dt.year

# Rename columns
pbi_df = pbi_df.rename(columns={
    'Temperature_F': 'Body_Temperature_F',
    'Target_Encoded': 'COVID_Positive_Flag'
})

# KPI columns
pbi_df['Total_Cases'] = 1
pbi_df['Total_COVID_Positive'] = pbi_df['COVID_Positive_Flag']
pbi_df['Total_Deceased'] = np.where(pbi_df['Patient_Outcome'] == 'Deceased', 1, 0)
pbi_df['Total_Recovered'] = np.where(pbi_df['Patient_Outcome'] == 'Recovered', 1, 0)

# Drop encoder keys
cols_to_drop = ['Gender_Encoded', 'Cough_Encoded', 'Breathing_Difficulty_Encoded', 
                'Heart_Disease_Encoded', 'Diabetes_Encoded']
pbi_df = pbi_df.drop(columns=[c for c in cols_to_drop if c in pbi_df.columns])

# Export dataset
output_pbi_path = "../data/powerbi_covid_dashboard_data.csv"
if not os.path.exists('../data'):
    output_pbi_path = "data/powerbi_covid_dashboard_data.csv"
    
pbi_df.to_csv(output_pbi_path, index=False)
print(f"Power BI dashboard dataset exported. Dimensions: {pbi_df.shape}")